<a href="https://colab.research.google.com/github/babbrian/vocabuddy-group-10/blob/main/VocaBuddy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Hello Google Colab")

Hello Google Colab


In [5]:
"""Minimal English / Chinese flashcards for Google Colab.

Paste this entire file into ONE Colab code cell and run it.
Or upload word_cards.py using Colab's Files sidebar, then run:
    %run /content/word_cards.py
Use %run, not !python: the buttons need the notebook's Python kernel.

Requires ipywidgets (normally available in Colab). If missing, run:
    %pip install ipywidgets

Cards autosave to cards.json in the current working directory.
Colab files are temporary: download a backup before ending your session.
Import accepts backups from this app and the original desktop version.

熟悉度模型：
  每張卡有 familiarity 欄位（預設 0）。
  按「認識 +1」→ 熟悉度 +1 → 帶權隨機時出現機率降低
  （權重 = 1 / (1 + familiarity)）。
  「重置此卡熟悉度」與「重置全部熟悉度」可把熟悉度歸零。
  熟悉度會一起存進 cards.json；匯入舊備份時缺少此欄位會自動補 0。
"""

import json
import random
from html import escape
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display


DATA_FILE = Path("cards.json").resolve()  # No __file__: works in notebook cells.


def parse_cards(text):
    """Validate a deck before accepting it from disk or an upload."""
    cards = json.loads(text)
    if not isinstance(cards, list) or any(
        not isinstance(card, dict)
        or any(
            not isinstance(card.get(key), str) or not card[key].strip()
            for key in ("english", "chinese")
        )
        for card in cards
    ):
        raise ValueError("Each card needs nonempty english and chinese text.")

    cleaned = []
    for card in cards:
        # [新增] 熟悉度：舊備份沒這個欄位就補 0；壞值（負數/非數字）也歸 0。
        raw = card.get("familiarity", 0)
        try:
            familiarity = max(0, int(raw))
        except (TypeError, ValueError):
            familiarity = 0
        cleaned.append({
            "english": card["english"].strip(),
            "chinese": card["chinese"].strip(),
            "familiarity": familiarity,
        })
    return cleaned


def load_cards(path=DATA_FILE):
    return parse_cards(path.read_text(encoding="utf-8")) if path.exists() else []


def save_cards(cards, path=DATA_FILE):
    """Replace the old deck only after the new file has been written."""
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(cards, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    temporary.replace(path)


class WordCardApp:
    def __init__(self, path=DATA_FILE):
        self.path = Path(path)
        self.cards = load_cards(self.path)
        self.index = 0
        self.show_chinese = False
        self.pending_delete = False

        self.counter = widgets.Label()
        self.card = widgets.HTML()
        self.status = widgets.Label()
        self.english = widgets.Text(description="English:", placeholder="apple")
        self.chinese = widgets.Text(description="中文:", placeholder="蘋果")

        # 導覽按鈕
        self.previous = widgets.Button(description="Previous")
        self.flip_button = widgets.Button(description="Flip / 翻面")
        self.next_button = widgets.Button(description="Next")
        self.random_button = widgets.Button(description="Random / 隨機")

        # 熟悉度相關
        self.know_button = widgets.Button(
            description="認識 +1", button_style="info"
        )
        self.reset_button = widgets.Button(description="重置此卡熟悉度")
        self.reset_all_button = widgets.Button(
            description="重置全部熟悉度", button_style="warning"
        )

        # 其他
        self.delete_button = widgets.Button(description="Delete", button_style="danger")
        self.add_button = widgets.Button(description="Add / 新增", button_style="success")
        self.download_button = widgets.Button(description="Download cards")
        self.upload = widgets.FileUpload(accept=".json", multiple=False,
                                         description="Import cards")
        self.output = widgets.Output()

        # Callbacks change the existing widgets; no separate window is needed.
        self.previous.on_click(lambda _: self.move(-1))
        self.next_button.on_click(lambda _: self.move(1))
        self.flip_button.on_click(self.flip)
        self.random_button.on_click(self.pick_random)
        self.know_button.on_click(self.mark_known)
        self.reset_button.on_click(self.reset_current_familiarity)
        self.reset_all_button.on_click(self.reset_all_familiarity)
        self.add_button.on_click(self.add_card)
        self.delete_button.on_click(self.delete_card)
        self.download_button.on_click(self.download_cards)
        self.upload.observe(self.import_cards, names="value")

        self.ui = widgets.VBox([
            widgets.HTML("<h3>English / 中文 Word Cards</h3>"),
            self.counter, self.card,
            widgets.Box(
                [self.previous, self.flip_button, self.next_button,
                 self.random_button],
                layout=widgets.Layout(display="flex", flex_flow="row wrap"),
            ),
            widgets.Box(
                [self.know_button, self.reset_button,
                 self.reset_all_button, self.delete_button],
                layout=widgets.Layout(display="flex", flex_flow="row wrap"),
            ),
            self.english, self.chinese, self.add_button, self.status,
            widgets.HBox([self.download_button, self.upload]),
            widgets.HTML(
                "<small>Autosaved for this session. Download a backup before leaving. "
                "Import adds cards and skips exact duplicates. "
                "按「認識 +1」會降低該卡在「隨機」中出現的機率。</small>"
            ),
            self.output,
        ], layout=widgets.Layout(width="100%", max_width="650px"))
        self.refresh()

    def refresh(self):
        self.pending_delete = False
        self.delete_button.description = "Delete"

        has_cards = bool(self.cards)
        for button in (
            self.previous, self.flip_button, self.next_button,
            self.delete_button, self.know_button, self.reset_button,
        ):
            button.disabled = not has_cards
        # 隨機只有在 2 張以上才有意義（1 張時只會顯示同一張）
        self.random_button.disabled = len(self.cards) < 2
        # 全部重置在 1 張以上就能用
        self.reset_all_button.disabled = not has_cards

        if self.cards:
            self.index %= len(self.cards)
            card = self.cards[self.index]
            side = "chinese" if self.show_chinese else "english"
            text = card[side]
            title = "中文" if self.show_chinese else "English"
            familiarity = card.get("familiarity", 0)
            self.counter.value = (
                f"Card {self.index + 1} / {len(self.cards)}"
                f"　·　熟悉度 {familiarity}"
            )
        else:
            title, text = "", "Add your first English / Chinese card below."
            self.counter.value = "0 cards"
        # Escape card text so punctuation and HTML-like text display literally.
        self.card.value = (
            '<div style="border:1px solid #aaa;border-radius:10px;padding:24px;'
            'min-height:120px;max-height:300px;overflow:auto;overflow-wrap:anywhere">'
            f'<small>{title}</small>'
            f'<div style="font-size:28px;white-space:pre-wrap">{escape(text)}</div></div>'
        )

    def commit(self, cards):
        try:
            save_cards(cards, self.path)
        except OSError as error:
            self.status.value = f"Could not save: {error}"
            return False
        self.cards = cards
        return True

    def add_card(self, _=None):
        english, chinese = self.english.value.strip(), self.chinese.value.strip()
        if not english or not chinese:
            self.status.value = "Enter both English and Chinese text."
            return
        new_card = {"english": english, "chinese": chinese, "familiarity": 0}
        if self.commit(self.cards + [new_card]):
            self.index = len(self.cards) - 1
            self.show_chinese = False
            self.english.value = self.chinese.value = ""
            self.status.value = "Card added and saved."
            self.refresh()

    def move(self, step):
        if self.cards:
            self.index = (self.index + step) % len(self.cards)
            self.show_chinese = False
            self.status.value = ""
            self.refresh()

    def pick_random(self, _=None):
        """帶權隨機：熟悉度越高、權重越低，越不容易被抽到。

        權重公式：weight = 1 / (1 + familiarity)
          familiarity 0 → 權重 1.0（最容易被抽到）
          familiarity 1 → 權重 0.5
          familiarity 5 → 權重約 0.17
        永遠排除目前這張，避免「按了看起來沒反應」。
        """
        if len(self.cards) < 2:
            return
        candidates = [i for i in range(len(self.cards)) if i != self.index]
        weights = [
            1.0 / (1.0 + self.cards[i].get("familiarity", 0))
            for i in candidates
        ]
        self.index = random.choices(candidates, weights=weights, k=1)[0]
        self.show_chinese = False
        self.status.value = "帶權隨機：越不熟的卡越容易出現。"
        self.refresh()

    def mark_known(self, _=None):
        """按「認識 +1」：把目前這張卡的熟悉度加 1，存檔。

        改副本再 commit，寫檔失敗時記憶體狀態不會被污染。
        """
        if not self.cards:
            return
        updated = [dict(card) for card in self.cards]
        updated[self.index]["familiarity"] = (
            updated[self.index].get("familiarity", 0) + 1
        )
        new_value = updated[self.index]["familiarity"]
        if self.commit(updated):
            self.status.value = (
                f"熟悉度 +1（目前 {new_value}）。"
                "之後按「隨機」時這張會更少出現。"
            )
            self.refresh()

    def reset_current_familiarity(self, _=None):
        """把目前這張卡的熟悉度歸零。"""
        if not self.cards:
            return
        updated = [dict(card) for card in self.cards]
        updated[self.index]["familiarity"] = 0
        if self.commit(updated):
            self.status.value = "已將這張卡的熟悉度歸零。"
            self.refresh()

    def reset_all_familiarity(self, _=None):
        """把所有卡片的熟悉度歸零。"""
        if not self.cards:
            return
        updated = [dict(card) for card in self.cards]
        for card in updated:
            card["familiarity"] = 0
        if self.commit(updated):
            self.status.value = "已將所有卡片的熟悉度歸零。"
            self.refresh()

    def flip(self, _=None):
        if self.cards:
            self.show_chinese = not self.show_chinese
            self.status.value = ""
            self.refresh()

    def delete_card(self, _=None):
        if not self.cards:
            return
        if not self.pending_delete:
            self.pending_delete = True
            self.delete_button.description = "Confirm delete"
            self.status.value = (
                "Click Confirm delete to remove this card; "
                "Flip / Next / Random cancels."
            )
            return
        if self.commit(self.cards[:self.index] + self.cards[self.index + 1:]):
            self.index = min(self.index, max(0, len(self.cards) - 1))
            self.show_chinese = False
            self.status.value = "Card deleted and saved."
            self.refresh()

    def import_cards(self, change):
        uploaded = change["new"]
        if not uploaded:
            return
        try:
            # ipywidgets 7 uses a dict; ipywidgets 8 uses a tuple of files.
            item = next(iter(uploaded.values())) if isinstance(uploaded, dict) else uploaded[0]
            incoming = parse_cards(bytes(item["content"]).decode("utf-8-sig"))
            merged = list(self.cards)
            seen = {(card["english"], card["chinese"]) for card in merged}
            for card in incoming:
                key = (card["english"], card["chinese"])
                if key not in seen:
                    merged.append(card)
                    seen.add(key)
            added = len(merged) - len(self.cards)
            if self.commit(merged):
                self.show_chinese = False
                self.status.value = f"Imported {added} new card(s) and saved."
                self.refresh()
        except (ValueError, KeyError, TypeError) as error:
            self.status.value = f"Import failed: {error}"
        finally:
            self.upload.value = {} if isinstance(self.upload.value, dict) else ()

    def download_cards(self, _=None):
        try:
            from google.colab import files
            save_cards(self.cards, self.path)
            with self.output:
                self.output.clear_output(wait=True)
                files.download(str(self.path))
            self.status.value = "Download requested. Keep the JSON file to import next time."
        except Exception as error:
            self.status.value = f"Download failed: {error}"


def main():
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass  # The basic app also works in Jupyter; Download is Colab-specific.
    try:
        app = WordCardApp()
    except (OSError, ValueError) as error:
        print(f"Could not load {DATA_FILE}: {error}")
        print("The existing file was left unchanged. Fix it or rename it, then rerun.")
        return None
    display(app.ui)
    return app


if __name__ == "__main__":
    # Rerunning the cell removes the previous UI to avoid two stale decks.
    old_app = globals().get("word_card_app")
    if old_app is not None:
        old_app.ui.close()
    word_card_app = main()

In [11]:
"""Minimal English / Chinese flashcards for Google Colab.

Paste this entire file into ONE Colab code cell and run it.
Use %run, not !python: the buttons need the notebook's Python kernel.

Cards autosave to cards.json in the current working directory.
Colab files are temporary: download a backup before ending your session.
Import accepts backups from this app and the original desktop version.

熟悉度模型：
  familiarity 欄位（預設 0）。按「認識 +1」+1，帶權隨機權重 = 1/(1+f)。
  「重置此卡熟悉度」/「重置全部熟悉度」可歸零。

測驗模式：
  10 秒內盡量答對，答錯一次即結束。
  題目為一個英文，配兩個中文選項（一正確一干擾）。
  題目純隨機，允許重複出現同一張卡。
  結束後停在得分畫面，需按「返回主畫面」才會回到瀏覽模式。

計時實作：
  倒數由瀏覽器端的 JavaScript setInterval 驅動（見 _start_js_countdown），
  時間到時透過 google.colab.kernel.invokeFunction 回呼 Python 端結算。
  這樣可以避免 asyncio / threading 在 Colab 裡更新 widget 失敗的問題。
"""

import json
import random
import time
from html import escape
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, Javascript


DATA_FILE = Path("cards.json").resolve()
QUIZ_DURATION = 10.0
QUIZ_TIMEOUT_CALLBACK = "word_card_quiz_timeout"  # JS → Python 的回呼名稱


def _add_class(widget, name):
    """把 CSS class 加到 widget 上（相容 ipywidgets 7 / 8）。"""
    if hasattr(widget, "add_class"):
        widget.add_class(name)
    else:
        widget._dom_classes = tuple(widget._dom_classes) + (name,)


def parse_cards(text):
    cards = json.loads(text)
    if not isinstance(cards, list) or any(
        not isinstance(card, dict)
        or any(
            not isinstance(card.get(key), str) or not card[key].strip()
            for key in ("english", "chinese")
        )
        for card in cards
    ):
        raise ValueError("Each card needs nonempty english and chinese text.")

    cleaned = []
    for card in cards:
        raw = card.get("familiarity", 0)
        try:
            familiarity = max(0, int(raw))
        except (TypeError, ValueError):
            familiarity = 0
        cleaned.append({
            "english": card["english"].strip(),
            "chinese": card["chinese"].strip(),
            "familiarity": familiarity,
        })
    return cleaned


def load_cards(path=DATA_FILE):
    return parse_cards(path.read_text(encoding="utf-8")) if path.exists() else []


def save_cards(cards, path=DATA_FILE):
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(cards, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    temporary.replace(path)


class WordCardApp:
    def __init__(self, path=DATA_FILE):
        self.path = Path(path)
        self.cards = load_cards(self.path)
        self.index = 0
        self.show_chinese = False
        self.pending_delete = False

        # ---- 測驗狀態 ----
        self.quiz_active = False
        self.quiz_token = 0
        self.quiz_score = 0
        self.quiz_deadline = 0.0
        self.quiz_answer_index = 0

        # ---- 主畫面零件 ----
        self.counter = widgets.Label()
        self.card = widgets.HTML()
        self.status = widgets.Label()
        self.english = widgets.Text(description="English:", placeholder="apple")
        self.chinese = widgets.Text(description="中文:", placeholder="蘋果")

        self.previous = widgets.Button(description="Previous")
        self.flip_button = widgets.Button(description="Flip / 翻面")
        self.next_button = widgets.Button(description="Next")
        self.random_button = widgets.Button(description="Random / 隨機")

        self.know_button = widgets.Button(description="認識 +1", button_style="info")
        self.reset_button = widgets.Button(description="重置此卡熟悉度")
        self.reset_all_button = widgets.Button(
            description="重置全部熟悉度", button_style="warning"
        )
        self.quiz_start_button = widgets.Button(
            description="開始 10 秒測驗", button_style="success"
        )
        self.delete_button = widgets.Button(description="Delete", button_style="danger")
        self.add_button = widgets.Button(description="Add / 新增", button_style="success")
        self.download_button = widgets.Button(description="Download cards")
        self.upload = widgets.FileUpload(accept=".json", multiple=False,
                                         description="Import cards")
        self.output = widgets.Output()

        # ---- 測驗畫面零件 ----
        # 計時標籤用 data-quiz-timer 屬性，讓 JS 能 querySelector 找到它
        self.quiz_timer_label = widgets.HTML(
            value='<span data-quiz-timer style="font-size:20px">⏱</span>'
        )
        self.quiz_score_label = widgets.HTML()
        self.quiz_question = widgets.HTML()
        self.quiz_option_buttons = []
        for i in range(2):
            btn = widgets.Button(
                description="",
                button_style="primary",
                layout=widgets.Layout(width="100%", height="95px"),
            )
            _add_class(btn, "quiz-option")
            btn.on_click(lambda _, idx=i: self._answer(idx))
            self.quiz_option_buttons.append(btn)
        self.quiz_quit_button = widgets.Button(
            description="放棄測驗", button_style="danger"
        )

        # ---- 得分畫面零件 ----
        self.quiz_result_label = widgets.HTML()
        self.quiz_return_button = widgets.Button(
            description="返回主畫面",
            button_style="success",
            layout=widgets.Layout(width="100%", height="60px"),
        )
        _add_class(self.quiz_return_button, "quiz-return")
        self.quiz_return_button.on_click(self._return_from_result)

        # ---- 綁定 ----
        self.previous.on_click(lambda _: self.move(-1))
        self.next_button.on_click(lambda _: self.move(1))
        self.flip_button.on_click(self.flip)
        self.random_button.on_click(self.pick_random)
        self.know_button.on_click(self.mark_known)
        self.reset_button.on_click(self.reset_current_familiarity)
        self.reset_all_button.on_click(self.reset_all_familiarity)
        self.quiz_start_button.on_click(self.start_quiz)
        self.quiz_quit_button.on_click(self.end_quiz_manually)
        self.add_button.on_click(self.add_card)
        self.delete_button.on_click(self.delete_card)
        self.download_button.on_click(self.download_cards)
        self.upload.observe(self.import_cards, names="value")

        # ---- 註冊 JS 超時回呼（Colab 專用）----
        self._register_js_timeout_callback()

        # ---- CSS ----
        self.css = widgets.HTML(
            "<style>"
            "button.quiz-option {"
            "  font-size: 30px !important;"
            "  font-weight: 600 !important;"
            "  line-height: 1.2 !important;"
            "}"
            "button.quiz-return { font-size: 20px !important; }"
            "</style>"
        )

        # ---- 主 UI ----
        self.main_ui = widgets.VBox([
            widgets.HTML("<h3>English / 中文 Word Cards</h3>"),
            self.counter, self.card,
            widgets.Box(
                [self.previous, self.flip_button, self.next_button,
                 self.random_button],
                layout=widgets.Layout(display="flex", flex_flow="row wrap"),
            ),
            widgets.Box(
                [self.know_button, self.reset_button,
                 self.reset_all_button, self.delete_button],
                layout=widgets.Layout(display="flex", flex_flow="row wrap"),
            ),
            self.quiz_start_button,
            self.english, self.chinese, self.add_button, self.status,
            widgets.HBox([self.download_button, self.upload]),
            widgets.HTML(
                "<small>Autosaved for this session. Download a backup before leaving. "
                "Import adds cards and skips exact duplicates. "
                "按「認識 +1」會降低該卡在「隨機」中出現的機率。</small>"
            ),
        ])

        # ---- 測驗 UI（預設隱藏）----
        self.quiz_ui = widgets.VBox(
            [
                widgets.HTML("<h3>⏱ 10 秒挑戰（答錯即結束）</h3>"),
                self.quiz_timer_label,
                self.quiz_score_label,
                self.quiz_question,
                self.quiz_option_buttons[0],
                self.quiz_option_buttons[1],
                self.quiz_quit_button,
            ],
            layout=widgets.Layout(display="none"),
        )

        # ---- 得分 UI（預設隱藏）----
        self.quiz_result_ui = widgets.VBox(
            [self.quiz_result_label, self.quiz_return_button],
            layout=widgets.Layout(display="none"),
        )

        self.ui = widgets.VBox(
            [self.css, self.main_ui, self.quiz_ui, self.quiz_result_ui, self.output],
            layout=widgets.Layout(width="100%", max_width="650px"),
        )
        self.refresh()

    # ---------- JS → Python 回呼註冊 ----------
    def _register_js_timeout_callback(self):
        """在 Colab 裡註冊可供前端 JS 呼叫的回呼。非 Colab 環境會安靜跳過。"""
        try:
            from google.colab import output as colab_output
        except ImportError:
            return
        try:
            colab_output.register_callback(
                QUIZ_TIMEOUT_CALLBACK, self._on_js_timeout
            )
        except Exception as error:
            # 重複註冊時某些版本會拋錯，忽略即可
            print(f"（JS 回呼註冊失敗，可忽略）{error}")

    def _on_js_timeout(self):
        """JS 倒數歸零時會呼叫這裡。只在測驗進行中才有效。"""
        if self.quiz_active:
            self._finalize_quiz("時間到")

    # ---------- 主畫面 ----------
    def refresh(self):
        self.pending_delete = False
        self.delete_button.description = "Delete"

        has_cards = bool(self.cards)
        for button in (
            self.previous, self.flip_button, self.next_button,
            self.delete_button, self.know_button, self.reset_button,
        ):
            button.disabled = not has_cards
        self.random_button.disabled = len(self.cards) < 2
        self.reset_all_button.disabled = not has_cards

        distinct_chinese = len({c["chinese"] for c in self.cards})
        self.quiz_start_button.disabled = distinct_chinese < 2

        if self.cards:
            self.index %= len(self.cards)
            card = self.cards[self.index]
            side = "chinese" if self.show_chinese else "english"
            text = card[side]
            title = "中文" if self.show_chinese else "English"
            familiarity = card.get("familiarity", 0)
            self.counter.value = (
                f"Card {self.index + 1} / {len(self.cards)}"
                f"　·　熟悉度 {familiarity}"
            )
        else:
            title, text = "", "Add your first English / Chinese card below."
            self.counter.value = "0 cards"

        self.card.value = (
            '<div style="border:1px solid #aaa;border-radius:10px;padding:24px;'
            'min-height:120px;max-height:300px;overflow:auto;overflow-wrap:anywhere">'
            f'<small>{title}</small>'
            f'<div style="font-size:28px;white-space:pre-wrap">{escape(text)}</div></div>'
        )

    def commit(self, cards):
        try:
            save_cards(cards, self.path)
        except OSError as error:
            self.status.value = f"Could not save: {error}"
            return False
        self.cards = cards
        return True

    def add_card(self, _=None):
        english, chinese = self.english.value.strip(), self.chinese.value.strip()
        if not english or not chinese:
            self.status.value = "Enter both English and Chinese text."
            return
        new_card = {"english": english, "chinese": chinese, "familiarity": 0}
        if self.commit(self.cards + [new_card]):
            self.index = len(self.cards) - 1
            self.show_chinese = False
            self.english.value = self.chinese.value = ""
            self.status.value = "Card added and saved."
            self.refresh()

    def move(self, step):
        if self.cards:
            self.index = (self.index + step) % len(self.cards)
            self.show_chinese = False
            self.status.value = ""
            self.refresh()

    def pick_random(self, _=None):
        if len(self.cards) < 2:
            return
        candidates = [i for i in range(len(self.cards)) if i != self.index]
        weights = [
            1.0 / (1.0 + self.cards[i].get("familiarity", 0))
            for i in candidates
        ]
        self.index = random.choices(candidates, weights=weights, k=1)[0]
        self.show_chinese = False
        self.status.value = "帶權隨機：越不熟的卡越容易出現。"
        self.refresh()

    def mark_known(self, _=None):
        if not self.cards:
            return
        updated = [dict(card) for card in self.cards]
        updated[self.index]["familiarity"] = (
            updated[self.index].get("familiarity", 0) + 1
        )
        new_value = updated[self.index]["familiarity"]
        if self.commit(updated):
            self.status.value = (
                f"熟悉度 +1（目前 {new_value}）。"
                "之後按「隨機」時這張會更少出現。"
            )
            self.refresh()

    def reset_current_familiarity(self, _=None):
        if not self.cards:
            return
        updated = [dict(card) for card in self.cards]
        updated[self.index]["familiarity"] = 0
        if self.commit(updated):
            self.status.value = "已將這張卡的熟悉度歸零。"
            self.refresh()

    def reset_all_familiarity(self, _=None):
        if not self.cards:
            return
        updated = [dict(card) for card in self.cards]
        for card in updated:
            card["familiarity"] = 0
        if self.commit(updated):
            self.status.value = "已將所有卡片的熟悉度歸零。"
            self.refresh()

    def flip(self, _=None):
        if self.cards:
            self.show_chinese = not self.show_chinese
            self.status.value = ""
            self.refresh()

    def delete_card(self, _=None):
        if not self.cards:
            return
        if not self.pending_delete:
            self.pending_delete = True
            self.delete_button.description = "Confirm delete"
            self.status.value = (
                "Click Confirm delete to remove this card; "
                "Flip / Next / Random cancels."
            )
            return
        if self.commit(self.cards[:self.index] + self.cards[self.index + 1:]):
            self.index = min(self.index, max(0, len(self.cards) - 1))
            self.show_chinese = False
            self.status.value = "Card deleted and saved."
            self.refresh()

    # ---------- 匯入 / 匯出 ----------
    def import_cards(self, change):
        uploaded = change["new"]
        if not uploaded:
            return
        try:
            item = next(iter(uploaded.values())) if isinstance(uploaded, dict) else uploaded[0]
            incoming = parse_cards(bytes(item["content"]).decode("utf-8-sig"))
            merged = list(self.cards)
            seen = {(card["english"], card["chinese"]) for card in merged}
            for card in incoming:
                key = (card["english"], card["chinese"])
                if key not in seen:
                    merged.append(card)
                    seen.add(key)
            added = len(merged) - len(self.cards)
            if self.commit(merged):
                self.show_chinese = False
                self.status.value = f"Imported {added} new card(s) and saved."
                self.refresh()
        except (ValueError, KeyError, TypeError) as error:
            self.status.value = f"Import failed: {error}"
        finally:
            self.upload.value = {} if isinstance(self.upload.value, dict) else ()

    def download_cards(self, _=None):
        try:
            from google.colab import files
            save_cards(self.cards, self.path)
            with self.output:
                self.output.clear_output(wait=True)
                files.download(str(self.path))
            self.status.value = "Download requested. Keep the JSON file to import next time."
        except Exception as error:
            self.status.value = f"Download failed: {error}"

    # ---------- 測驗模式 ----------
    def start_quiz(self, _=None):
        if len({c["chinese"] for c in self.cards}) < 2:
            self.status.value = "至少需要 2 張不同中文的卡片才能開始測驗。"
            return

        self.quiz_active = True
        self.quiz_token += 1
        token = self.quiz_token
        self.quiz_score = 0
        self.quiz_deadline = time.time() + QUIZ_DURATION

        self.quiz_score_label.value = '<b>答對 0 題</b>'
        # 初始倒數文字：先寫入，讓 JS 接手後續更新
        self.quiz_timer_label.value = (
            f'<span data-quiz-timer style="font-size:20px">'
            f'⏱ 剩餘 {QUIZ_DURATION:.1f} 秒</span>'
        )

        self.main_ui.layout.display = "none"
        self.quiz_result_ui.layout.display = "none"
        self.quiz_ui.layout.display = ""
        self.status.value = ""

        self._next_question()
        self._start_js_countdown(token)

    def _start_js_countdown(self, token):
        """用瀏覽器端 JS 跑倒數；時間到時回呼 Python。

        為什麼不用 Python 的 threading / asyncio？
        因為在 Colab 裡，從非主執行緒更新 widget 常常被靜默丟棄，
        導致計時器不會動，甚至把 comm 通道塞爆而卡死。
        JS setInterval 完全在前端跑，穩定又不會影響 Python 端。
        """
        script = f"""
        (function() {{
            if (window._wordCardQuizTimer) {{
                clearInterval(window._wordCardQuizTimer);
                window._wordCardQuizTimer = null;
            }}
            var startTime = Date.now();
            var duration = {int(QUIZ_DURATION * 1000)};
            var token = {token};
            var timeoutFired = false;

            function findTimerEls() {{
                return document.querySelectorAll('[data-quiz-timer]');
            }}

            window._wordCardQuizTimer = setInterval(function() {{
                var remaining = Math.max(
                    0, (duration - (Date.now() - startTime)) / 1000
                );
                var els = findTimerEls();
                for (var i = 0; i < els.length; i++) {{
                    els[i].textContent = '⏱ 剩餘 ' + remaining.toFixed(1) + ' 秒';
                }}
                if (!timeoutFired && (Date.now() - startTime) >= duration) {{
                    timeoutFired = true;
                    clearInterval(window._wordCardQuizTimer);
                    window._wordCardQuizTimer = null;
                    try {{
                        google.colab.kernel.invokeFunction(
                            '{QUIZ_TIMEOUT_CALLBACK}', [], {{}}
                        );
                    }} catch (e) {{
                        console.error('quiz timeout callback failed', e);
                    }}
                }}
            }}, 100);
        }})();
        """
        try:
            display(Javascript(script))
        except Exception as error:
            print(f"（JS 倒數啟動失敗）{error}")

    def _stop_js_countdown(self):
        """停止前端倒數計時器（結束測驗時呼叫）。"""
        try:
            display(Javascript(
                "if (window._wordCardQuizTimer) {"
                "  clearInterval(window._wordCardQuizTimer);"
                "  window._wordCardQuizTimer = null;"
                "}"
            ))
        except Exception:
            pass

    def _next_question(self):
        """隨機出一題：題目純隨機，允許同一張卡連續出現。"""
        if not self.quiz_active or not self.cards:
            return

        q_index = random.randrange(len(self.cards))
        correct = self.cards[q_index]["chinese"]
        english_text = self.cards[q_index]["english"]

        distractor_pool = [
            c["chinese"] for i, c in enumerate(self.cards)
            if i != q_index and c["chinese"] != correct
        ]
        if not distractor_pool:
            self._finalize_quiz("沒有足夠不同的中文可作選項")
            return

        wrong = random.choice(distractor_pool)
        options = [correct, wrong]
        random.shuffle(options)
        self.quiz_answer_index = options.index(correct)

        self.quiz_question.value = (
            '<div style="border:1px solid #aaa;border-radius:10px;'
            'padding:20px;text-align:center;min-height:80px">'
            f'<div style="font-size:36px">{escape(english_text)}</div>'
            '</div>'
        )
        for i, btn in enumerate(self.quiz_option_buttons):
            btn.description = options[i]
            btn.disabled = False

    def _answer(self, chosen_index):
        if not self.quiz_active:
            return

        # 搶答邊界：若其實已超時，直接結算
        if time.time() > self.quiz_deadline:
            self._finalize_quiz("時間到")
            return

        # 立刻鎖住選項，避免快速連點重複計分
        for btn in self.quiz_option_buttons:
            btn.disabled = True

        if chosen_index == self.quiz_answer_index:
            self.quiz_score += 1
            self.quiz_score_label.value = f'<b>答對 {self.quiz_score} 題</b>'
            self._next_question()
        else:
            self._finalize_quiz("答錯了")

    def _finalize_quiz(self, reason):
        """結束測驗，切到得分畫面。多路徑呼叫時只會執行一次。"""
        if not self.quiz_active:
            return
        self.quiz_active = False
        score = self.quiz_score
        self._stop_js_countdown()

        for btn in self.quiz_option_buttons:
            btn.disabled = True

        self.quiz_result_label.value = (
            '<div style="text-align:center;padding:30px 10px;'
            'border:1px solid #aaa;border-radius:10px">'
            f'<div style="font-size:22px;color:#888">{escape(reason)}</div>'
            f'<div style="font-size:52px;font-weight:700;margin:18px 0">'
            f'{score} 題</div>'
            '<div style="font-size:16px;color:#666">最終得分</div>'
            '</div>'
        )
        self.quiz_ui.layout.display = "none"
        self.quiz_result_ui.layout.display = ""

    def _return_from_result(self, _=None):
        self.quiz_result_ui.layout.display = "none"
        self.main_ui.layout.display = ""
        self.status.value = f"上次測驗得分：{self.quiz_score} 題。"
        self.refresh()

    def end_quiz_manually(self, _=None):
        self._finalize_quiz("放棄測驗")


def main():
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
    try:
        app = WordCardApp()
    except (OSError, ValueError) as error:
        print(f"Could not load {DATA_FILE}: {error}")
        print("The existing file was left unchanged. Fix it or rename it, then rerun.")
        return None
    display(app.ui)
    return app


if __name__ == "__main__":
    old_app = globals().get("word_card_app")
    if old_app is not None:
        old_app.ui.close()
    word_card_app = main()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
"""Minimal English / Chinese flashcards for Google Colab.

Paste this entire file into ONE Colab code cell and run it.
Use %run, not !python: the buttons need the notebook's Python kernel.

Cards autosave to cards.json in the current working directory.
Colab files are temporary: download a backup before ending your session.
Import accepts backups from this app and the original desktop version.

熟悉度模型：
  familiarity 欄位（預設 0）。按「認識 +1」+1，帶權隨機權重 = 1/(1+f)。
  「重置此卡熟悉度」/「重置全部熟悉度」可歸零。

測驗模式：
  10 秒內盡量答對，答錯一次即結束。
  題目為一個英文，配兩個中文選項（一正確一干擾）。
  題目純隨機，允許重複出現同一張卡。
  結束後停在得分畫面，需按「返回主畫面」才會回到瀏覽模式。

計時實作：
  倒數由瀏覽器端 JavaScript setInterval 驅動（見 _start_js_countdown），
  時間到時透過 google.colab.kernel.invokeFunction 回呼 Python 端結算。
  Python 端用 quiz_token 驗證，避免舊 timer 誤觸發。

預設競程卡片：
  按「匯入預設競程卡片」會把 DEFAULT_CARDS 合併進現有卡組，跳過重複。
"""

import json
import random
import time
from html import escape
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, Javascript


DATA_FILE = Path("cards.json").resolve()
QUIZ_DURATION = 10.0
QUIZ_TIMEOUT_CALLBACK = "word_card_quiz_timeout"  # JS → Python 的回呼名稱


# 預設競程卡片（按「匯入預設競程卡片」時合併進卡組，跳過重複）
DEFAULT_CARDS = [
    {"english": "SCC", "chinese": "強連通分量", "familiarity": 0},
    {"english": "BCC", "chinese": "雙聯通分量", "familiarity": 0},
    {"english": "DP",  "chinese": "動態規劃",   "familiarity": 0},
    {"english": "BIT", "chinese": "二元索引樹", "familiarity": 0},
    {"english": "DSU", "chinese": "並查集",     "familiarity": 0},
]


def _add_class(widget, name):
    """把 CSS class 加到 widget 上（相容 ipywidgets 7 / 8）。"""
    if hasattr(widget, "add_class"):
        widget.add_class(name)
    else:
        widget._dom_classes = tuple(widget._dom_classes) + (name,)


def parse_cards(text):
    cards = json.loads(text)
    if not isinstance(cards, list) or any(
        not isinstance(card, dict)
        or any(
            not isinstance(card.get(key), str) or not card[key].strip()
            for key in ("english", "chinese")
        )
        for card in cards
    ):
        raise ValueError("Each card needs nonempty english and chinese text.")

    cleaned = []
    for card in cards:
        raw = card.get("familiarity", 0)
        try:
            familiarity = max(0, int(raw))
        except (TypeError, ValueError):
            familiarity = 0
        cleaned.append({
            "english": card["english"].strip(),
            "chinese": card["chinese"].strip(),
            "familiarity": familiarity,
        })
    return cleaned


def load_cards(path=DATA_FILE):
    return parse_cards(path.read_text(encoding="utf-8")) if path.exists() else []


def save_cards(cards, path=DATA_FILE):
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(cards, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    temporary.replace(path)


class WordCardApp:
    def __init__(self, path=DATA_FILE):
        self.path = Path(path)
        self.cards = load_cards(self.path)
        self.index = 0
        self.show_chinese = False
        self.pending_delete = False
        self.pending_delete_all = False

        # ---- 測驗狀態 ----
        self.quiz_active = False
        self.quiz_token = 0
        self.quiz_score = 0
        self.quiz_deadline = 0.0
        self.quiz_answer_index = 0

        # ---- 主畫面零件 ----
        self.counter = widgets.Label()
        self.card = widgets.HTML()
        self.status = widgets.Label()
        self.english = widgets.Text(description="English:", placeholder="apple")
        self.chinese = widgets.Text(description="中文:", placeholder="蘋果")

        self.previous = widgets.Button(description="Previous")
        self.flip_button = widgets.Button(description="Flip / 翻面")
        self.next_button = widgets.Button(description="Next")
        self.random_button = widgets.Button(description="Random / 隨機")

        self.know_button = widgets.Button(description="認識 +1", button_style="info")
        self.reset_button = widgets.Button(description="重置此卡熟悉度")
        self.reset_all_button = widgets.Button(
            description="重置全部熟悉度", button_style="warning"
        )
        self.delete_button = widgets.Button(description="Delete", button_style="danger")

        # [新增] 匯入預設 / 刪除全部
        self.import_default_button = widgets.Button(
            description="匯入預設競程卡片", button_style="primary"
        )
        self.delete_all_button = widgets.Button(
            description="刪除所有卡片", button_style="danger"
        )

        self.quiz_start_button = widgets.Button(
            description="開始 10 秒測驗", button_style="success"
        )
        self.add_button = widgets.Button(description="Add / 新增", button_style="success")
        self.download_button = widgets.Button(description="Download cards")
        self.upload = widgets.FileUpload(accept=".json", multiple=False,
                                         description="Import cards")
        self.output = widgets.Output()

        # ---- 測驗畫面零件 ----
        self.quiz_timer_label = widgets.HTML(
            value='<span data-quiz-timer style="font-size:20px">⏱</span>'
        )
        self.quiz_score_label = widgets.HTML()
        self.quiz_question = widgets.HTML()
        self.quiz_option_buttons = []
        for i in range(2):
            btn = widgets.Button(
                description="",
                button_style="primary",
                layout=widgets.Layout(width="100%", height="95px"),
            )
            _add_class(btn, "quiz-option")
            btn.on_click(lambda _, idx=i: self._answer(idx))
            self.quiz_option_buttons.append(btn)
        self.quiz_quit_button = widgets.Button(
            description="放棄測驗", button_style="danger"
        )

        # ---- 得分畫面零件 ----
        self.quiz_result_label = widgets.HTML()
        self.quiz_return_button = widgets.Button(
            description="返回主畫面",
            button_style="success",
            layout=widgets.Layout(width="100%", height="60px"),
        )
        _add_class(self.quiz_return_button, "quiz-return")
        self.quiz_return_button.on_click(self._return_from_result)

        # ---- 綁定 ----
        self.previous.on_click(lambda _: self.move(-1))
        self.next_button.on_click(lambda _: self.move(1))
        self.flip_button.on_click(self.flip)
        self.random_button.on_click(self.pick_random)
        self.know_button.on_click(self.mark_known)
        self.reset_button.on_click(self.reset_current_familiarity)
        self.reset_all_button.on_click(self.reset_all_familiarity)
        self.delete_button.on_click(self.delete_card)
        self.import_default_button.on_click(self.import_default_cards)   # [新增]
        self.delete_all_button.on_click(self.delete_all_cards)           # [新增]
        self.quiz_start_button.on_click(self.start_quiz)
        self.quiz_quit_button.on_click(self.end_quiz_manually)
        self.add_button.on_click(self.add_card)
        self.download_button.on_click(self.download_cards)
        self.upload.observe(self.import_cards, names="value")

        # ---- 註冊 JS 超時回呼（Colab 專用）----
        self._register_js_timeout_callback()

        # ---- CSS ----
        self.css = widgets.HTML(
            "<style>"
            "button.quiz-option {"
            "  font-size: 30px !important;"
            "  font-weight: 600 !important;"
            "  line-height: 1.2 !important;"
            "}"
            "button.quiz-return { font-size: 20px !important; }"
            "</style>"
        )

        # ---- 主 UI ----
        self.main_ui = widgets.VBox([
            widgets.HTML("<h3>English / 中文 Word Cards</h3>"),
            self.counter, self.card,
            widgets.Box(
                [self.previous, self.flip_button, self.next_button,
                 self.random_button],
                layout=widgets.Layout(display="flex", flex_flow="row wrap"),
            ),
            widgets.Box(
                [self.know_button, self.reset_button,
                 self.reset_all_button, self.delete_button],
                layout=widgets.Layout(display="flex", flex_flow="row wrap"),
            ),
            # [新增] 兩個新按鈕單獨一行，避免與上面擠在一起
            widgets.Box(
                [self.import_default_button, self.delete_all_button],
                layout=widgets.Layout(display="flex", flex_flow="row wrap"),
            ),
            self.quiz_start_button,
            self.english, self.chinese, self.add_button, self.status,
            widgets.HBox([self.download_button, self.upload]),
            widgets.HTML(
                "<small>Autosaved for this session. Download a backup before leaving. "
                "Import adds cards and skips exact duplicates. "
                "按「認識 +1」會降低該卡在「隨機」中出現的機率。</small>"
            ),
        ])

        # ---- 測驗 UI（預設隱藏）----
        self.quiz_ui = widgets.VBox(
            [
                widgets.HTML("<h3>⏱ 10 秒挑戰（答錯即結束）</h3>"),
                self.quiz_timer_label,
                self.quiz_score_label,
                self.quiz_question,
                self.quiz_option_buttons[0],
                self.quiz_option_buttons[1],
                self.quiz_quit_button,
            ],
            layout=widgets.Layout(display="none"),
        )

        # ---- 得分 UI（預設隱藏）----
        self.quiz_result_ui = widgets.VBox(
            [self.quiz_result_label, self.quiz_return_button],
            layout=widgets.Layout(display="none"),
        )

        self.ui = widgets.VBox(
            [self.css, self.main_ui, self.quiz_ui, self.quiz_result_ui, self.output],
            layout=widgets.Layout(width="100%", max_width="650px"),
        )
        self.refresh()

    # ---------- JS → Python 回呼註冊 ----------
    def _register_js_timeout_callback(self):
        try:
            from google.colab import output as colab_output
        except ImportError:
            return
        try:
            colab_output.register_callback(
                QUIZ_TIMEOUT_CALLBACK, self._on_js_timeout
            )
        except Exception as error:
            print(f"（JS 回呼註冊失敗，可忽略）{error}")

    def _on_js_timeout(self, token=None):
        """JS 倒數歸零時會呼叫這裡。只在「當前這一輪」才有效。"""
        if not self.quiz_active:
            return
        try:
            if token is not None and int(token) != self.quiz_token:
                return
        except (TypeError, ValueError):
            pass
        self._finalize_quiz("時間到")

    # ---------- 主畫面 ----------
    def refresh(self):
        self.pending_delete = False
        self.pending_delete_all = False
        self.delete_button.description = "Delete"
        self.delete_all_button.description = "刪除所有卡片"

        has_cards = bool(self.cards)
        for button in (
            self.previous, self.flip_button, self.next_button,
            self.delete_button, self.know_button, self.reset_button,
        ):
            button.disabled = not has_cards
        self.random_button.disabled = len(self.cards) < 2
        self.reset_all_button.disabled = not has_cards
        self.delete_all_button.disabled = not has_cards    # [新增]
        # import_default_button 永遠可用（就算卡組是空的也能匯入）

        distinct_chinese = len({c["chinese"] for c in self.cards})
        self.quiz_start_button.disabled = distinct_chinese < 2

        if self.cards:
            self.index %= len(self.cards)
            card = self.cards[self.index]
            side = "chinese" if self.show_chinese else "english"
            text = card[side]
            title = "中文" if self.show_chinese else "English"
            familiarity = card.get("familiarity", 0)
            self.counter.value = (
                f"Card {self.index + 1} / {len(self.cards)}"
                f"　·　熟悉度 {familiarity}"
            )
        else:
            title, text = "", "Add your first English / Chinese card below."
            self.counter.value = "0 cards"

        self.card.value = (
            '<div style="border:1px solid #aaa;border-radius:10px;padding:24px;'
            'min-height:120px;max-height:300px;overflow:auto;overflow-wrap:anywhere">'
            f'<small>{title}</small>'
            f'<div style="font-size:28px;white-space:pre-wrap">{escape(text)}</div></div>'
        )

    def commit(self, cards):
        try:
            save_cards(cards, self.path)
        except OSError as error:
            self.status.value = f"Could not save: {error}"
            return False
        self.cards = cards
        return True

    def add_card(self, _=None):
        english, chinese = self.english.value.strip(), self.chinese.value.strip()
        if not english or not chinese:
            self.status.value = "Enter both English and Chinese text."
            return
        new_card = {"english": english, "chinese": chinese, "familiarity": 0}
        if self.commit(self.cards + [new_card]):
            self.index = len(self.cards) - 1
            self.show_chinese = False
            self.english.value = self.chinese.value = ""
            self.status.value = "Card added and saved."
            self.refresh()

    def move(self, step):
        if self.cards:
            self.index = (self.index + step) % len(self.cards)
            self.show_chinese = False
            self.status.value = ""
            self.refresh()

    def pick_random(self, _=None):
        if len(self.cards) < 2:
            return
        candidates = [i for i in range(len(self.cards)) if i != self.index]
        weights = [
            1.0 / (1.0 + self.cards[i].get("familiarity", 0))
            for i in candidates
        ]
        self.index = random.choices(candidates, weights=weights, k=1)[0]
        self.show_chinese = False
        self.status.value = "帶權隨機：越不熟的卡越容易出現。"
        self.refresh()

    def mark_known(self, _=None):
        if not self.cards:
            return
        updated = [dict(card) for card in self.cards]
        updated[self.index]["familiarity"] = (
            updated[self.index].get("familiarity", 0) + 1
        )
        new_value = updated[self.index]["familiarity"]
        if self.commit(updated):
            self.status.value = (
                f"熟悉度 +1（目前 {new_value}）。"
                "之後按「隨機」時這張會更少出現。"
            )
            self.refresh()

    def reset_current_familiarity(self, _=None):
        if not self.cards:
            return
        updated = [dict(card) for card in self.cards]
        updated[self.index]["familiarity"] = 0
        if self.commit(updated):
            self.status.value = "已將這張卡的熟悉度歸零。"
            self.refresh()

    def reset_all_familiarity(self, _=None):
        if not self.cards:
            return
        updated = [dict(card) for card in self.cards]
        for card in updated:
            card["familiarity"] = 0
        if self.commit(updated):
            self.status.value = "已將所有卡片的熟悉度歸零。"
            self.refresh()

    def flip(self, _=None):
        if self.cards:
            self.show_chinese = not self.show_chinese
            self.status.value = ""
            self.refresh()

    def delete_card(self, _=None):
        if not self.cards:
            return
        if not self.pending_delete:
            self.pending_delete = True
            self.delete_button.description = "Confirm delete"
            self.status.value = (
                "Click Confirm delete to remove this card; "
                "Flip / Next / Random cancels."
            )
            return
        if self.commit(self.cards[:self.index] + self.cards[self.index + 1:]):
            self.index = min(self.index, max(0, len(self.cards) - 1))
            self.show_chinese = False
            self.status.value = "Card deleted and saved."
            self.refresh()

    # ---------- [新增] 匯入預設競程卡 ----------
    def import_default_cards(self, _=None):
        """把 DEFAULT_CARDS 合併進卡組，跳過已存在的 (english, chinese)。"""
        merged = list(self.cards)
        seen = {(c["english"], c["chinese"]) for c in merged}
        added = 0
        for card in DEFAULT_CARDS:
            key = (card["english"], card["chinese"])
            if key not in seen:
                merged.append(dict(card))
                seen.add(key)
                added += 1

        if added == 0:
            self.status.value = "預設競程卡片已全部在卡組中，未新增。"
            return
        if self.commit(merged):
            self.show_chinese = False
            self.status.value = f"已匯入 {added} 張預設競程卡片並存檔。"
            self.refresh()

    # ---------- [新增] 刪除所有卡片（兩段式確認）----------
    def delete_all_cards(self, _=None):
        if not self.cards:
            return
        if not self.pending_delete_all:
            self.pending_delete_all = True
            self.delete_all_button.description = "確認刪除全部"
            self.status.value = (
                "這會刪除全部卡片且無法復原。"
                "再按一次「確認刪除全部」以執行；其他動作會取消。"
            )
            return
        if self.commit([]):
            self.index = 0
            self.show_chinese = False
            self.status.value = "已刪除所有卡片並存檔。"
            self.refresh()

    # ---------- 匯入 / 匯出 ----------
    def import_cards(self, change):
        uploaded = change["new"]
        if not uploaded:
            return
        try:
            item = next(iter(uploaded.values())) if isinstance(uploaded, dict) else uploaded[0]
            incoming = parse_cards(bytes(item["content"]).decode("utf-8-sig"))
            merged = list(self.cards)
            seen = {(card["english"], card["chinese"]) for card in merged}
            for card in incoming:
                key = (card["english"], card["chinese"])
                if key not in seen:
                    merged.append(card)
                    seen.add(key)
            added = len(merged) - len(self.cards)
            if self.commit(merged):
                self.show_chinese = False
                self.status.value = f"Imported {added} new card(s) and saved."
                self.refresh()
        except (ValueError, KeyError, TypeError) as error:
            self.status.value = f"Import failed: {error}"
        finally:
            self.upload.value = {} if isinstance(self.upload.value, dict) else ()

    def download_cards(self, _=None):
        try:
            from google.colab import files
            save_cards(self.cards, self.path)
            with self.output:
                self.output.clear_output(wait=True)
                files.download(str(self.path))
            self.status.value = "Download requested. Keep the JSON file to import next time."
        except Exception as error:
            self.status.value = f"Download failed: {error}"

    # ---------- 測驗模式 ----------
    def start_quiz(self, _=None):
        if len({c["chinese"] for c in self.cards}) < 2:
            self.status.value = "至少需要 2 張不同中文的卡片才能開始測驗。"
            return

        self.quiz_active = True
        self.quiz_token += 1
        token = self.quiz_token
        self.quiz_score = 0
        self.quiz_deadline = time.time() + QUIZ_DURATION

        self.quiz_score_label.value = '<b>答對 0 題</b>'
        self.quiz_timer_label.value = (
            f'<span data-quiz-timer style="font-size:20px">'
            f'⏱ 剩餘 {QUIZ_DURATION:.1f} 秒</span>'
        )

        self.main_ui.layout.display = "none"
        self.quiz_result_ui.layout.display = "none"
        self.quiz_ui.layout.display = ""
        self.status.value = ""

        self._next_question()
        self._start_js_countdown(token)

    def _start_js_countdown(self, token):
        """用瀏覽器端 JS 跑倒數；時間到時回呼 Python。"""
        script = f"""
        (function() {{
            var startTime = Date.now();
            var duration = {int(QUIZ_DURATION * 1000)};
            var token = {token};
            var timeoutFired = false;

            function findTimerEls() {{
                return document.querySelectorAll('[data-quiz-timer]');
            }}

            var timer = setInterval(function() {{
                var remaining = Math.max(
                    0, (duration - (Date.now() - startTime)) / 1000
                );
                var els = findTimerEls();
                for (var i = 0; i < els.length; i++) {{
                    els[i].textContent = '⏱ 剩餘 ' + remaining.toFixed(1) + ' 秒';
                }}
                if (!timeoutFired && (Date.now() - startTime) >= duration) {{
                    timeoutFired = true;
                    clearInterval(timer);
                    try {{
                        google.colab.kernel.invokeFunction(
                            '{QUIZ_TIMEOUT_CALLBACK}', [token], {{}}
                        );
                    }} catch (e) {{
                        console.error('quiz timeout callback failed', e);
                    }}
                }}
            }}, 100);
        }})();
        """
        try:
            display(Javascript(script))
        except Exception as error:
            print(f"（JS 倒數啟動失敗）{error}")

    def _next_question(self):
        """隨機出一題：題目純隨機，允許同一張卡連續出現。"""
        if not self.quiz_active or not self.cards:
            return

        q_index = random.randrange(len(self.cards))
        correct = self.cards[q_index]["chinese"]
        english_text = self.cards[q_index]["english"]

        distractor_pool = [
            c["chinese"] for i, c in enumerate(self.cards)
            if i != q_index and c["chinese"] != correct
        ]
        if not distractor_pool:
            self._finalize_quiz("沒有足夠不同的中文可作選項")
            return

        wrong = random.choice(distractor_pool)
        options = [correct, wrong]
        random.shuffle(options)
        self.quiz_answer_index = options.index(correct)

        self.quiz_question.value = (
            '<div style="border:1px solid #aaa;border-radius:10px;'
            'padding:20px;text-align:center;min-height:80px">'
            f'<div style="font-size:36px">{escape(english_text)}</div>'
            '</div>'
        )
        for i, btn in enumerate(self.quiz_option_buttons):
            btn.description = options[i]
            btn.disabled = False

    def _answer(self, chosen_index):
        if not self.quiz_active:
            return

        # 搶答邊界：若其實已超時，直接結算
        if time.time() > self.quiz_deadline:
            self._finalize_quiz("時間到")
            return

        for btn in self.quiz_option_buttons:
            btn.disabled = True

        if chosen_index == self.quiz_answer_index:
            self.quiz_score += 1
            self.quiz_score_label.value = f'<b>答對 {self.quiz_score} 題</b>'
            self._next_question()
        else:
            self._finalize_quiz("答錯了")

    def _finalize_quiz(self, reason):
        """結束測驗，切到得分畫面。多路徑呼叫時只會執行一次。"""
        if not self.quiz_active:
            return
        self.quiz_active = False
        score = self.quiz_score

        for btn in self.quiz_option_buttons:
            btn.disabled = True

        self.quiz_result_label.value = (
            '<div style="text-align:center;padding:30px 10px;'
            'border:1px solid #aaa;border-radius:10px">'
            f'<div style="font-size:22px;color:#888">{escape(reason)}</div>'
            f'<div style="font-size:52px;font-weight:700;margin:18px 0">'
            f'{score} 題</div>'
            '<div style="font-size:16px;color:#666">最終得分</div>'
            '</div>'
        )
        self.quiz_ui.layout.display = "none"
        self.quiz_result_ui.layout.display = ""

    def _return_from_result(self, _=None):
        self.quiz_result_ui.layout.display = "none"
        self.main_ui.layout.display = ""
        self.status.value = f"上次測驗得分：{self.quiz_score} 題。"
        self.refresh()

    def end_quiz_manually(self, _=None):
        self._finalize_quiz("放棄測驗")


def main():
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
    try:
        app = WordCardApp()
    except (OSError, ValueError) as error:
        print(f"Could not load {DATA_FILE}: {error}")
        print("The existing file was left unchanged. Fix it or rename it, then rerun.")
        return None
    display(app.ui)
    return app


if __name__ == "__main__":
    old_app = globals().get("word_card_app")
    if old_app is not None:
        old_app.ui.close()
    word_card_app = main()

<IPython.core.display.Javascript object>